# Influenta factorilor demografici si economici asupra emisiilor CO2 la nivel global

## Introducere

Acest notebook prezintă analiza relației dintre factorii economici și emisiile de CO2 la nivel global.

Scopul este identificarea variabilelor care influențează emisiile și evidențierea unor tipare economice relevante.

## Importuri

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## Incărcare date

In [ ]:
df = pd.read_excel("../data/CO2_emissions_factors.xlsx")
df.head()

## Tratarea valorilor lipsă

In [ ]:
df.isnull().sum()

numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df.isnull().sum()

Valorile lipsă au fost tratate prin imputare cu mediana pentru toate variabilele numerice.
Această metodă este robustă la valori extreme și păstrează distribuția reală a datelor,
fiind preferată în analiza econometrică față de media aritmetică.

## Detectarea valorilor extreme

In [ ]:
sns.boxplot(data=df[numeric_cols])
plt.xticks(rotation=45)
plt.show()

Analiza boxplot indică prezența unor valori extreme, în special pentru variabile economice precum FDI.

Aceste valori reflectă diferențe semnificative între țări și pot influența rezultatele modelelor statistice.

## Eliminarea variabilei categorice

In [ ]:
df = df.drop(columns=["Country"])

Coloana "Country" a fost eliminată deoarece nu are semnificație numerică și nu este relevantă pentru modelarea statistică.

## Transformări econometrice

In [ ]:
df["log_CO2"] = np.log(df["CO2_emissions"] + 1)
df["log_GDP"] = np.log(df["gdp_pc_ppp"] + 1)
df["log_population"] = np.log(df["total_population"] + 1)
df["asinh_fdi"] = np.arcsinh(df["fdi"])

scaler = StandardScaler()
percent_cols = ["urban_population", "trade_in_services", "renewable_energy"]
df[percent_cols] = scaler.fit_transform(df[percent_cols])

Transformările aplicate au redus asimetria distribuțiilor și impactul valorilor extreme.

Variabilele devin mai apropiate de o distribuție normală, ceea ce îmbunătățește stabilitatea și acuratețea modelului.

## Gruparea datelor

In [ ]:
df["GDP_Group"] = pd.qcut(df["gdp_pc_ppp"], 4)

df.groupby("GDP_Group", observed=False)["CO2_emissions"].mean()

Rezultatele arată că emisiile de CO2 cresc odată cu nivelul PIB-ului.

Această relație sugerează că economiile mai dezvoltate au un impact mai ridicat asupra mediului.

## Regresie liniară multiplă

In [ ]:
X = df[[
    "log_GDP",
    "log_population",
    "asinh_fdi",
    "urban_population",
    "trade_in_services",
    "renewable_energy"
]]

y = df["log_CO2"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

X_train_sm = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_sm).fit()

model.summary()

### Interpretare

Modelul de regresie evidențiază o relație semnificativă între factorii economici și emisiile de CO2.

PIB-ul are un efect pozitiv asupra emisiilor, în timp ce variabile precum energia regenerabilă contribuie la reducerea acestora.

Modelul explică o proporție semnificativă din variația emisiilor (R²), ceea ce indică o performantă bună a modelului.


## Clusterizare

In [ ]:
X_cluster = df[["log_GDP", "log_CO2"]]

kmeans = KMeans(n_clusters=2)
df["Cluster"] = kmeans.fit_predict(X_cluster)

sns.scatterplot(x=df["log_GDP"], y=df["log_CO2"], hue=df["Cluster"])
plt.show()

### Interpretare

Clusterizarea evidențiază existența unor grupuri distincte de țări în funcție de nivelul PIB și al emisiilor de CO2.

Rezultatele sugerează diferențe structurale între economii, confirmând heterogenitatea datelor la nivel global.

## Analiza performanței modelului pe clustere

### Predicții + eroare


In [ ]:
X_full = sm.add_constant(X)
df["Predicted_log_CO2"] = model.predict(X_full)

df["Predicted_CO2"] = np.exp(df["Predicted_log_CO2"]) - 1

df["Error"] = df["CO2_emissions"] - df["Predicted_CO2"]

### Analiză pe clustere

In [ ]:
cluster_analysis = df.groupby("Cluster").agg({
    "CO2_emissions": "mean",
    "Predicted_CO2": "mean",
    "Error": ["mean", "std"]
})

print(cluster_analysis)

### Grafic REAL vs ESTIMAT

In [ ]:
sns.scatterplot(
    x=df["CO2_emissions"],
    y=df["Predicted_CO2"],
    hue=df["Cluster"]
)

plt.plot(
    [df["CO2_emissions"].min(), df["CO2_emissions"].max()],
    [df["CO2_emissions"].min(), df["CO2_emissions"].max()],
    color='red'
)

plt.title("CO2 real vs estimat")
plt.show()

### Boxplot erori (FOARTE IMPORTANT)

In [ ]:
sns.boxplot(
    x=df["Cluster"],
    y=df["Error"]
)

plt.title("Distribuția erorilor pe clustere")
plt.show()

### Interpretare

Modelul de regresie nu performează uniform pentru toate grupurile de țări.

Pentru clusterul caracterizat prin niveluri mai reduse și mai omogene ale emisiilor,
valorile estimate sunt apropiate de cele reale, iar erorile sunt mici și compacte.

În schimb, pentru clusterul asociat cu emisii ridicate, se observă o dispersie mai mare a valorilor,
iar modelul înregistrează erori semnificative, inclusiv outlieri.

Acest rezultat sugerează existența unor relații non-lineare sau a unor factori suplimentari
care influențează emisiile de CO2 în economiile mai dezvoltate.

## Concluzii

Analiza realizată evidențiază existența unei relații semnificative între dezvoltarea economică și nivelul emisiilor de CO2.

Rezultatele modelului de regresie arată că PIB-ul are un impact pozitiv asupra emisiilor, în timp ce utilizarea energiei regenerabile contribuie la reducerea acestora.

Clusterizarea a confirmat existența unor grupuri distincte de țări, diferențiate în funcție de nivelul de dezvoltare economică și de intensitatea emisiilor.

Aceste rezultate sugerează că dezvoltarea economică este asociată cu un impact crescut asupra mediului, subliniind necesitatea unor politici sustenabile.

## Implicații practice

Rezultatele analizei pot fi utilizate pentru:

- formularea politicilor publice de mediu
- evaluarea impactului dezvoltării economice asupra emisiilor
- identificarea țărilor cu risc ridicat de poluare
- prioritizarea investițiilor în energie regenerabilă

În ansamblu, rezultatele indică faptul că dezvoltarea economică este asociată cu un impact semnificativ asupra mediului,
iar reducerea emisiilor depinde în mare măsură de tranziția către surse de energie sustenabile.

## Limitări și direcții viitoare

Modelul de regresie liniară nu captează complet relațiile non-lineare din date, în special pentru economiile cu emisii ridicate.

Modele de tip Machine Learning (ex: Random Forest, Gradient Boosting) ar putea îmbunătăți performanța și acuratețea predicțiilor.